# ⚙️ Feature Engineering: Tabular Data (Jurnal)
Tujuan dari notebook ini adalah melakukan transformasi data mentah menjadi format yang siap diolah oleh model *Machine Learning*. Langkah yang dilakukan meliputi pembuatan fitur IMT, *Encoding*, *Standardization*, dan *Splitting* dataset.

## Import Library

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Path setup
DATA_INPUT_PATH = '../data/processed/dataset_stunting_clean.csv'
DATA_OUTPUT_DIR = '../data/data_fe'

print("✅ Library & Path terkonfigurasi.")

✅ Library & Path terkonfigurasi.


## Load Dataset

In [ ]:
print("⏳ Memuat dataset...")
df = pd.read_csv(DATA_INPUT_PATH)
df.head()

⏳ Memuat dataset...


,id_anak,jenis_kelamin,usia_bulan,berat_badan,tinggi_badan,status_bb_u,zscore_bb_u,status_tb_u,zscore_tb_u,status_bb_tb,zscore_bb_tb
0,1,Perempuan,54.0,13.2,97.5,Normal,-1.94,Stunting,-2.11,Normal,-0.95
1,2,Laki-laki,44.0,12.0,92.0,Normal,-1.92,Stunting,-2.22,Normal,-0.88
2,3,Laki-laki,57.0,14.0,97.0,Normal,-1.90,Stunting,-2.58,Normal,-0.48
3,4,Laki-laki,26.0,11.0,79.0,Normal,-1.15,Stunting,-3.11,Normal,0.68
4,5,Perempuan,59.0,14.6,98.0,Normal,-1.66,Stunting,-2.49,Normal,-0.18


## Feature Creation (IMT)

In [7]:
print("⚙️ Menghitung fitur IMT (Indeks Massa Tubuh)...")
# Rumus: Berat (kg) / (Tinggi (m) ^ 2)
df['imt'] = df['berat_badan'] / ((df['tinggi_badan'] / 100) ** 2)

print(f"✅ Berhasil menambah fitur 'imt'. Sampel data:\n{df[['berat_badan', 'tinggi_badan', 'imt']].head()}")

⚙️ Menghitung fitur IMT (Indeks Massa Tubuh)...
✅ Berhasil menambah fitur 'imt'. Sampel data:
   berat_badan  tinggi_badan        imt
0         13.2          97.5  13.885602
1         12.0          92.0  14.177694
2         14.0          97.0  14.879371
3         11.0          79.0  17.625381
4         14.6          98.0  15.201999


## Feature Encoding

In [8]:
print("⚙️ Mengubah variabel kategorikal menjadi numerik...")

le_jk = LabelEncoder()
df['jenis_kelamin_encoded'] = le_jk.fit_transform(df['jenis_kelamin'])

le_target = LabelEncoder()
df['status_tb_u_encoded'] = le_target.fit_transform(df['status_tb_u'])

print(f"Mapping JK: {dict(zip(le_jk.classes_, le_jk.transform(le_jk.classes_)))}")
print(f"Mapping Status: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

⚙️ Mengubah variabel kategorikal menjadi numerik...
Mapping JK: {'Laki-laki': np.int64(0), 'Perempuan': np.int64(1)}
Mapping Status: {'Stunting': np.int64(0), 'Tidak Stunting': np.int64(1)}


## Train-Test Split

In [9]:
fitur_pilihan = ['usia_bulan', 'jenis_kelamin_encoded', 'tinggi_badan', 'berat_badan', 'imt']
X = df[fitur_pilihan]
y = df['status_tb_u_encoded']

# Split 80:20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✅ Split selesai: Train ({X_train.shape[0]}), Test ({X_test.shape[0]})")

✅ Split selesai: Train (32052), Test (8014)


## Feature Scaling

In [10]:
print("⚙️ Scaling data dengan StandardScaler...")
scaler = StandardScaler()

# Fit & Transform hanya di train untuk menghindari Data Leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_final = pd.DataFrame(X_train_scaled, columns=fitur_pilihan)
X_test_final = pd.DataFrame(X_test_scaled, columns=fitur_pilihan)

print("✅ Scaling selesai.")

⚙️ Scaling data dengan StandardScaler...
✅ Scaling selesai.


## Save Final Features

In [11]:
os.makedirs(DATA_OUTPUT_DIR, exist_ok=True)

# Gabung kembali target untuk disimpan
X_train_final['target'] = y_train.values
X_test_final['target'] = y_test.values

X_train_final.to_csv(os.path.join(DATA_OUTPUT_DIR, 'tabular_train_features.csv'), index=False)
X_test_final.to_csv(os.path.join(DATA_OUTPUT_DIR, 'tabular_test_features.csv'), index=False)

print("🏁 SUCCESS: File fitur siap untuk Modeling!")

🏁 SUCCESS: File fitur siap untuk Modeling!
